# Problem: Train a 3D CNN network for segmenting CT images
### Problem Statement
You are tasked with employing and evaluating a 3D CNN model in Pytorch for semantic segmentation on synthetically generated CT images. 
Your goal is to review the input and label data shapes. Next, define a MedCNN model class with a `forward` method that emulates a encode-decoder architecture with appropriate input and output channels based on the input shapes.   

### Requirements
1. **Implement** a MedCNN model class: a 2D ResNet18 backbone encodes every slice (it downsamples width and height by 32), `Conv3d` layers then mix the slice features along the depth axis, and `ConvTranspose3d` layers upsample back to the input resolution.
2. **Define** the soft Dice loss for the problem.
3. **Perform** transfer learning from a ResNet18 - a common strategy for custom architectures.
4. **Train** the model for 5 epochs.
### Constraints
- Use `Pytorch` in-built convolution layers
- Ensure there is a segmentation head at the end of the network that outputs one probability per voxel, in the **same layout as the masks**: `[B, D, 1, W, H]`


<details>
  <summary>💡 Hint</summary>
  - Strip off the `Avgpooling` and linear layers from ResNet18 using `list(resnet_model.children())[:-2]`
  <br>
  - [Conv3D](https://pytorch.org/docs/stable/generated/torch.nn.Conv3d.html)
  <br>
  - [ConvTranspose3D](https://pytorch.org/docs/stable/generated/torch.nn.ConvTranspose3d.html)
  <br>
  - [Forum discussion on model.children](https://discuss.pytorch.org/t/module-children-vs-module-modules/4551)
</details>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

In [ ]:
# Generate synthetic CT-scan data (batches, slices, RGB) and associated segmentation masks.
# Kept small so a full-batch forward/backward pass through ResNet18 fits in CPU / Colab memory
# (width and height must be multiples of 32, the backbone's downsampling factor).
torch.manual_seed(42)
batch = 4
num_slices = 8
channels = 3
width = 128
height = 128

ct_images = torch.randn(size=(batch, num_slices, channels, width, height))
segmentation_masks = (torch.randn(size=(batch, num_slices, 1, width, height))>0).float()

print(f"CT images (train examples) shape: {ct_images.shape}")
print(f"Segmentation binary masks (labels) shape: {segmentation_masks.shape}")

In [ ]:
# Define the MedCNN class and its forward method
class MedCNN(nn.Module):
    def __init__(self, backbone, out_channel=1):
        super(MedCNN, self).__init__()
        self.backbone = backbone
        
        # 3D convolutions over the stacked slice features. The 2D backbone has already
        # downsampled W and H by 32; these layers mix information across neighbouring
        # slices (the depth axis) and reduce the channels 512 -> 64.
        self.conv1 = nn.Conv3d(512, 64, kernel_size=(3, 3, 3), padding=1)
        self.conv2 = nn.Conv3d(64, 64, kernel_size=(3, 3, 3), padding=1)
        
        # Upsample W and H by 4 * 8 = 32; kernel/stride 1 on the depth axis leaves the slices alone
        self.conv_transpose1 = nn.ConvTranspose3d(64, 32, kernel_size=(1, 4, 4), stride=(1, 4, 4))
        self.conv_transpose2 = nn.ConvTranspose3d(32, 16, kernel_size=(1, 8, 8), stride=(1, 8, 8))
        
        # Segmentation head: 1x1x1 convolution from 16 channels to out_channel
        self.final_conv = nn.Conv3d(16, out_channel, kernel_size=1)
        self.relu = nn.ReLU()

    def forward(self, x):
        b, d, c, w, h = x.size() #Input size: [B, D, C, W, H]
        print(f"Input shape [B, D, C, W, H]: {b, d, c, w, h}")
        
        x = x.reshape(b*d, c, w, h) #Input to ResNet 2D conv layers [B*D, C, W, H]
        features = self.backbone(x)
        print(f"ResNet output shape[B*D, C, W, H]: {features.shape}")
        
        _, new_c, new_w, new_h = features.size()
        x = features.view(b, d, new_c, new_w, new_h) #[B, D, C, W, H]
        x = torch.permute(x, (0, 2, 1, 3, 4)) #rearrange for 3DConv layers [B, C, D, W, H]
        print(f"Reshape Resnet output for 3DConv #1 [B, C, D, W, H]: {x.shape}")
        
        #3D convolutions across slices
        x = self.relu(self.conv1(x))
        print(f"Output shape 3D Conv #1: {x.shape}")
        x = self.relu(self.conv2(x))
        print(f"Output shape 3D Conv #2: {x.shape}")
        
        #Upsampling
        x = self.relu(self.conv_transpose1(x))
        print(f"Output shape 3D Transposed Conv #1: {x.shape}")
        x = self.relu(self.conv_transpose2(x))
        print(f"Output shape 3D Transposed Conv #2: {x.shape}")

        #final segmentation: one probability per voxel, [B, out_channel, D, W, H]
        x = torch.sigmoid(self.final_conv(x))
        
        #back to the mask layout [B, D, out_channel, W, H]. Without this, pred * labels in the
        #Dice loss silently broadcasts [B, 1, D, W, H] x [B, D, 1, W, H] -> [B, D, D, W, H]
        x = torch.permute(x, (0, 2, 1, 3, 4))
        print(f"Final shape [B, D, 1, W, H]: {x.shape}")
        
        return x

In [ ]:
def compute_dice_loss(pred, labels, eps=1e-8):
    '''
    Soft Dice LOSS = 1 - Dice coefficient.

    The coefficient rises as overlap improves, so the loss must be its
    complement - otherwise minimising it trains the model away from the
    target.

    Args
    pred:   [B, D, 1, W, H] predicted probabilities in [0, 1]
    labels: [B, D, 1, W, H] binary ground truth

    Returns
    dice_loss: scalar in [0, 1]; 0 means perfect overlap
    '''
    # Guard against silent broadcasting: [B, 1, D, W, H] * [B, D, 1, W, H] is legal
    # PyTorch but multiplies every slice with every other slice.
    assert pred.shape == labels.shape, \
        f"pred {tuple(pred.shape)} and labels {tuple(labels.shape)} must have the same layout"
    numerator = 2 * torch.sum(pred * labels)
    denominator = torch.sum(pred) + torch.sum(labels) + eps
    dice_coefficient = numerator / denominator
    return 1 - dice_coefficient


In [ ]:
# Define resnet as the backbone removing the last two layers (avgpool and fc)
resnet_model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
resnet_model = nn.Sequential(*list(resnet_model.children())[:-2])

model = MedCNN(backbone=resnet_model)

optimizer = optim.Adam(model.parameters(), lr=0.01)

In [ ]:
epochs = 5
for epoch in range(epochs):
    optimizer.zero_grad()
    pred = model(ct_images)
    loss = compute_dice_loss(pred, segmentation_masks)
    loss.backward()
    optimizer.step()
    print(f"Loss at epoch {epoch}: {loss}")